# Encrypting and Decrypting Secret Messages

Have you ever wanted to pass a note to a friend without a nosy classmate being able to read it along the way?

That is what **cryptography** is for. It turns a readable message into scrambled nonsense that only the right person can turn back. In this notebook you will do it yourself in Python, using the same `cryptography` library that real apps rely on.

# What is Encryption?

Three words to know:

- **Plaintext** - your original, readable message.
- **Ciphertext** - the scrambled version, safe to send anywhere.
- **Key** - a secret value that controls the scrambling. Without the right key, the ciphertext is useless.

**Encrypting** turns plaintext into ciphertext. **Decrypting** turns it back. The security comes entirely from keeping the *key* secret and never from hiding *how* the encryption works.

# Symmetric Key: One Shared Secret

The simplest kind of encryption uses **the same key to encrypt and to decrypt**. This is called *symmetric* encryption. You and your friend each need a copy of that one key, and you both keep it secret from everyone else.

Python's `cryptography` library has a beginner-friendly recipe for this called **Fernet**. Run the cell below to import it and make a key.

> The first cell you run will take a few seconds while Python downloads the `cryptography` library. After that it is instant.

In [ ]:
from cryptography.fernet import Fernet

# A key is just 44 random characters. Generate a fresh one:
key = Fernet.generate_key()
print(key)

Now that we have generated a key, we can use it to encrypt a message using the `cipher.encrypt` method:

In [ ]:
cipher = Fernet(key)

message = "Meet me by the lockers at 3pm"
ciphertext = cipher.encrypt(message.encode())
print(ciphertext)

That jumble is your **ciphertext**. It is only letters, numbers, and a couple of symbols, so you can safely paste it into a chat message, an email, or even a QR code.

Anyone who intercepts it sees noise. The only way to get the message back is to have the key.

## Bytes vs. text

Notice the `b` in front of `b'Meet me...'`, and the `.encode()` call. The `cryptography` library works with **bytes** (raw data), not ordinary text **strings**.

- `.encode()` turns a string into bytes, ready to encrypt.
- `.decode()` turns bytes back into a readable string after decrypting.

In [ ]:
decrypted = cipher.decrypt(ciphertext)
print(decrypted.decode())

The key used to decrypt the message has to be the same key used to encrypt the message. See what happens when an attacker tries to decrypt your message with a different key...

In [ ]:
from cryptography.fernet import InvalidToken

# Someone who does not have your key tries to read the message:
attacker_key = Fernet.generate_key()
attacker_cipher = Fernet(attacker_key)

try:
  attacker_cipher.decrypt(ciphertext)
except InvalidToken:
  print("Decryption failed — wrong key!")

## Try it yourself

Change the message and the key below, then watch the ciphertext update. This cell re-runs automatically whenever you change a field.

For two people to exchange secret messages they must use the **exact same key**. Change `KEY` and you get different ciphertext — and the original recipient can no longer read it.

In [ ]:
#@title Encrypt your own message
MESSAGE = "You are invited to my party!" #@param
KEY = "VAHsQJm2fnwCEhVpdJ-nP36YkzpPakwLc1m6uscytCI=" #@param

try:
  my_cipher = Fernet(KEY.encode())
  secret = my_cipher.encrypt(MESSAGE.encode())
  print("Ciphertext:")
  print(secret.decode())
  print()
  print("Decrypted again:")
  print(my_cipher.decrypt(secret).decode())
except Exception:
  print("That does not look like a valid Fernet key.")
  print("A key is 44 characters and ends with '='. Use the one from the first cell.")

# The Key-Sharing Problem

Symmetric encryption has one big catch: **how do you and your friend agree on the key in the first place?**

If you send the key in a message, anyone watching that message now has it too. You could whisper it in person - but how about if your friend is in another city, or if you are building a website that strangers need to send data to securely?

This is the problem that **asymmetric** encryption solves.

# Asymmetric Key: A Public Lock and a Private Key

Asymmetric encryption (also called **public-key** encryption) uses **two** keys that belong together:

- A **public key** - you can give this to anyone. It can only *lock* messages.
- A **private key** - you keep this one secret. It is the only thing that can *unlock* messages that were locked with your public key.

Think of the public key as an open padlock you mail to your friends. They put a note in a box, snap your padlock shut, and send it back. Only you can open it.

Let's make a public and private key pair. This is slower than making a symmetric key, so it might take a few seconds on your machine.

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes

private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

print("Key pair created.")

In [ ]:
# This padding is an advanced setting that prevents attackers from deducing patterns from ciphertext.
SCHEME = padding.OAEP(
  mgf=padding.MGF1(algorithm=hashes.SHA256()),
  algorithm=hashes.SHA256(),
  label=None,
)

note = "You're invited to my party on Saturday!"
locked = public_key.encrypt(note.encode(), SCHEME)

print(len(locked), "bytes of ciphertext")
print(locked[:40], "...")

In [ ]:
opened = private_key.decrypt(locked, SCHEME)
print(opened.decode())

Because the public key can only lock messages, it is safe to publish anywhere — on your website, in your email signature, in a QR code. There is no key-sharing problem any more.

The same pair of keys can also prove *who sent a message*: if you scramble something with your **private** key, anyone can check it with your **public** key and know it really came from you. This is called a **digital signature**, and it is how your browser knows a website is genuine.

# Recap

- **Symmetric** (Fernet): one shared secret key. Fast and simple — but both people have to get that key safely.
- **Asymmetric** (RSA): a public key anyone can use to send *you* a message, and a private key only you hold. Solves key sharing, but slower.
- Real systems use **both**: asymmetric encryption to agree on a symmetric key, then fast symmetric encryption for the rest of the conversation.

# Check Your Understanding

Think of a situation in your own life where you would want to encrypt something - maybe a message, a file, a diary. What would you be protecting, and who from? Would symmetric or asymmetric encryption fit better?

---
